# 🛍️ The Souled Store — Category Analytics
### Associate Category Analyst Portfolio | Jayesh
---
**Run each cell one by one using `Shift + Enter`**

## Step 1 — Install Libraries (run once)

In [ ]:
# Run this only if you get an import error
# !pip install pandas numpy matplotlib seaborn

## Step 2 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.family"] = "sans-serif"

print("✅ Libraries loaded!")

## Step 3 — Load Your CSV Files
> Make sure all 6 CSV files are uploaded in the same folder as this notebook

In [ ]:
orders    = pd.read_csv("orders.csv",                  parse_dates=["order_date"])
products  = pd.read_csv("products.csv")
customers = pd.read_csv("customers.csv")
inventory = pd.read_csv("inventory.csv")
marketing = pd.read_csv("marketing_spend.csv")
channel_r = pd.read_csv("channel_revenue_monthly.csv", parse_dates=["date"])

# Delivered orders only — used in all sales analysis
delivered = orders[orders["order_status"] == "Delivered"].copy()

print("✅ All files loaded!")
print(f"   Orders (delivered) : {len(delivered):,} rows")
print(f"   Products           : {len(products):,} SKUs")
print(f"   Customers          : {len(customers):,} rows")
print(f"   Inventory          : {len(inventory):,} rows")
print(f"   Marketing          : {len(marketing):,} rows")

## Step 4 — Preview Data

In [ ]:
print("ORDERS — first 5 rows")
orders.head()

In [ ]:
print("PRODUCTS — first 5 rows")
products.head()

In [ ]:
print("INVENTORY — first 5 rows")
inventory.head()

In [ ]:
print("CUSTOMERS — first 5 rows")
customers.head()

## 📊 KPI Summary — 2022 to 2024

In [ ]:
# Key metrics per year
rows = []
for yr in [2022, 2023, 2024]:
    d    = delivered[delivered["year"] == yr]
    prev = delivered[delivered["year"] == yr-1]
    rev  = d["revenue_inr"].sum() / 100000
    prev_rev = prev["revenue_inr"].sum() / 100000 if len(prev) > 0 else rev
    rows.append({
        "Year":              yr,
        "Revenue (₹ Lakhs)": round(rev, 1),
        "YoY Growth %":      round((rev - prev_rev) / prev_rev * 100, 1),
        "Total Orders":      d["order_id"].nunique(),
        "Avg Order Value":   round(d["selling_price"].mean(), 0),
        "Units Sold":        int(d["quantity"].sum()),
        "Gross Margin %":    round(d["gross_margin_pct"].mean(), 1),
        "Avg Discount %":    round(d["discount_pct"].mean(), 1),
        "Return Rate %":     round(orders[orders["year"]==yr]["is_returned"].mean()*100, 1),
        "Unique Buyers":     d["customer_id"].nunique(),
    })

kpi_df = pd.DataFrame(rows).set_index("Year")
kpi_df

---
## 📈 Chart 1 — Monthly Revenue Trend (2022–2024)

In [ ]:
# Colour palette
RED    = "#E31837"
DARK   = "#1A1A2E"
GOLD   = "#FFB800"
GREEN  = "#27AE60"
BLUE   = "#2980B9"

CAT_COLORS = {
    "Marvel":"#E31837", "DC":"#1A1A2E", "Friends":"#FFB800",
    "Anime":"#6A0DAD", "Harry Potter":"#7B3F00", "Music":"#2C3E50",
    "Movies & TV":"#E74C3C", "Sports":"#27AE60",
    "Basics":"#95A5A6", "Original Art":"#F39C12",
}
CH_COLORS = {
    "D2C Website":"#E31837", "Amazon":"#FF9900", "Flipkart":"#2874F0",
    "Myntra":"#FF3F6C", "Nykaa Fashion":"#FC2779",
    "Offline (Studio Store)":"#2C3E50",
}
monthly = (delivered
    .groupby(["year","month","month_name"])["revenue_inr"].sum()
    .reset_index().sort_values(["year","month"]))
monthly["rev_L"] = monthly["revenue_inr"] / 100000

fig, ax = plt.subplots(figsize=(13, 5))
styles = {2022:("--","o","#95A5A6"), 2023:("-","s",DARK), 2024:("-","D",RED)}

for yr, (ls, mk, col) in styles.items():
    g = monthly[monthly["year"]==yr]
    ax.plot(g["month"], g["rev_L"], ls=ls, marker=mk,
            linewidth=2, markersize=6, color=col, label=str(yr))
    peak = g.loc[g["rev_L"].idxmax()]
    ax.annotate(f"₹{peak['rev_L']:.0f}L", xy=(peak["month"], peak["rev_L"]),
                xytext=(0,10), textcoords="offset points",
                ha="center", fontsize=8, color=col, fontweight="bold")

ax.axvspan(9.5, 12.5, alpha=0.07, color=RED,  label="Festive (Oct–Dec)")
ax.axvspan(1.5,  2.5, alpha=0.05, color=GOLD, label="Valentine / Sale")
ax.set_xticks(range(1,13))
ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun",
                    "Jul","Aug","Sep","Oct","Nov","Dec"])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"₹{x:.0f}L"))
ax.set_title("Monthly Revenue Trend — The Souled Store (2022–2024)",
             fontsize=13, fontweight="bold", color=DARK)
ax.set_ylabel("Revenue (₹ Lakhs)"); ax.set_xlabel("Month")
ax.legend(fontsize=9, loc="upper left")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()

print("💡 Insight: Oct–Dec (Festive Q3) is the peak quarter every year.")

## 📊 Chart 2 — Category Revenue vs Gross Margin %

In [ ]:
cat = (delivered.groupby("category")
       .agg(rev=("revenue_inr","sum"), gm=("gross_margin_pct","mean"))
       .reset_index().sort_values("rev", ascending=False))
cat["rev_L"] = cat["rev"] / 100000

fig, ax1 = plt.subplots(figsize=(13, 5))
colors = [CAT_COLORS.get(c,"#888") for c in cat["category"]]
bars = ax1.bar(cat["category"], cat["rev_L"], color=colors, alpha=0.85, width=0.6)
for b in bars:
    ax1.text(b.get_x()+b.get_width()/2, b.get_height()+0.05,
             f"₹{b.get_height():.1f}L", ha="center", fontsize=8, fontweight="bold")
ax1.set_ylabel("Revenue (₹ Lakhs)"); ax1.tick_params(axis="x", rotation=30)

ax2 = ax1.twinx()
ax2.plot(cat["category"], cat["gm"], color=GOLD, marker="D",
         linewidth=2.5, markersize=8, label="Gross Margin %")
ax2.axhline(50, ls="--", color=GOLD, lw=1, alpha=0.5)
ax2.set_ylabel("Gross Margin %", color=GOLD); ax2.set_ylim(0, 85)
ax1.set_title("Category Revenue vs Gross Margin %",
              fontsize=13, fontweight="bold", color=DARK)
ax1.spines[["top","right"]].set_visible(False)
ax2.legend(loc="upper right")
plt.tight_layout(); plt.show()

print("💡 Insight: Licensed categories (Marvel, Anime) drive revenue but compress margin. Basics & Original Art have the best margin.")

## 🛒 Chart 3 — Channel Revenue Mix by Year

In [ ]:
ch = (delivered.groupby(["year","channel"])["revenue_inr"].sum()
      .reset_index().pivot(index="year", columns="channel", values="revenue_inr").fillna(0))
ch_pct = ch.div(ch.sum(axis=1), axis=0) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Absolute
bot = np.zeros(len(ch))
for i, col in enumerate(ch.columns):
    clr = list(CH_COLORS.values())[i % len(CH_COLORS)]
    ax1.bar(ch.index.astype(str), ch[col]/100000, bottom=bot/100000,
            label=col, color=clr, alpha=0.88)
    bot += ch[col].values
ax1.set_title("Revenue (₹ Lakhs)", fontweight="bold", color=DARK)
ax1.set_ylabel("₹ Lakhs"); ax1.spines[["top","right"]].set_visible(False)
ax1.legend(fontsize=7, bbox_to_anchor=(1,1))

# Percentage
bot2 = np.zeros(len(ch_pct))
for i, col in enumerate(ch_pct.columns):
    clr = list(CH_COLORS.values())[i % len(CH_COLORS)]
    bars2 = ax2.bar(ch_pct.index.astype(str), ch_pct[col], bottom=bot2,
                    color=clr, alpha=0.88)
    for bar, v in zip(bars2, ch_pct[col].values):
        if v > 8:
            ax2.text(bar.get_x()+bar.get_width()/2,
                     bar.get_y()+bar.get_height()/2,
                     f"{v:.0f}%", ha="center", va="center",
                     fontsize=8, fontweight="bold", color="white")
    bot2 += ch_pct[col].values
ax2.set_title("Channel Share %", fontweight="bold", color=DARK)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:.0f}%"))
ax2.spines[["top","right"]].set_visible(False)

fig.suptitle("Channel Revenue Split", fontsize=13, fontweight="bold", color=DARK)
plt.tight_layout(); plt.show()

print("💡 Insight: D2C Website = highest margin channel. Monitor if marketplace share is growing — it means more discount leakage.")

## 🌡️ Chart 4 — Sell-Through Rate Heatmap (Category × Product Type)

In [ ]:
types = ["Classic T-Shirt","Oversized T-Shirt","Hoodie",
         "Sweatshirt","Joggers","Shorts","Crop Top","Tote Bag"]
pivot = (inventory.groupby(["category","product_type"])["sell_through_pct"]
         .mean().unstack().round(1))
pivot = pivot[[c for c in types if c in pivot.columns]]

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="RdYlGn",
            vmin=0, vmax=80, linewidths=0.5, linecolor="#eee",
            cbar_kws={"label":"Sell-Through %","shrink":0.7}, ax=ax)
ax.set_title("Sell-Through % — Category × Product Type\n(Green = Fast | Red = Slow)",
             fontsize=12, fontweight="bold", color=DARK)
ax.tick_params(axis="x", rotation=35, labelsize=9)
ax.tick_params(axis="y", rotation=0, labelsize=9)
plt.tight_layout(); plt.show()

print("💡 Insight: Use this as your buying guide — green = increase intake, red = reduce or plan markdown.")

## 📦 Chart 5 — Inventory Health Status

In [ ]:
counts = inventory["stock_health_status"].value_counts()
colors_inv = {"Healthy":"#27AE60","Normal":"#3498DB",
              "Low Stock":"#F39C12","Overstocked":"#E74C3C","Stockout":"#8E44AD"}
clrs = [colors_inv.get(s,"#aaa") for s in counts.index]

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, pcts = ax.pie(counts.values, labels=counts.index, colors=clrs,
                              autopct="%1.1f%%", pctdistance=0.82, startangle=140,
                              wedgeprops={"linewidth":2,"edgecolor":"white"})
for p in pcts: p.set_fontsize(9); p.set_fontweight("bold"); p.set_color("white")

circle = plt.Circle((0,0), 0.60, fc="white")
ax.add_patch(circle)
ax.text(0, 0.06, "Stock\nHealth", ha="center", va="center",
        fontsize=12, fontweight="bold", color=DARK)
ax.text(0, -0.22, f"{len(inventory)} SKU-Sizes", ha="center", fontsize=9, color="#777")
ax.set_title("Inventory Health Distribution", fontsize=12, fontweight="bold", color=DARK)
plt.tight_layout(); plt.show()

over_pct = (counts.get("Overstocked",0) / len(inventory) * 100)
print(f"💡 Insight: {over_pct:.1f}% of SKU-sizes are overstocked — markdown or intake reduction needed.")

## 📏 Chart 6 — Size Curve Analysis

In [ ]:
size_order = ["XS","S","M","L","XL","XXL"]
sz = (inventory.groupby("size")
      .agg(sold=("units_sold","sum"), st=("sell_through_pct","mean"))
      .reindex(size_order).reset_index())

fig, ax1 = plt.subplots(figsize=(9, 5))
bar_clrs = [RED if s in ["M","L"] else "#95A5A6" for s in sz["size"]]
bars = ax1.bar(sz["size"], sz["sold"], color=bar_clrs, alpha=0.85, width=0.5)
for b in bars:
    ax1.text(b.get_x()+b.get_width()/2, b.get_height()+3,
             str(int(b.get_height())), ha="center", fontsize=9, fontweight="bold")
ax1.set_ylabel("Units Sold"); ax1.set_xlabel("Size")

ax2 = ax1.twinx()
ax2.plot(sz["size"], sz["st"], color=DARK, marker="o", linewidth=2.5, markersize=9)
for i, (s, v) in enumerate(zip(sz["size"], sz["st"])):
    ax2.text(i, v+0.2, f"{v:.0f}%", ha="center", fontsize=9,
             color=DARK, fontweight="bold")
ax2.set_ylabel("Avg Sell-Through %", color=DARK)
ax2.set_ylim(0, sz["st"].max() * 2)

ax1.set_title("Size Curve — Units Sold & Sell-Through %",
              fontsize=12, fontweight="bold", color=DARK)
ax1.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()

print("💡 Insight: M & L are core sizes. If XS/XXL sell-through is low, reduce their ratio in next purchase order.")

## 🏆 Chart 7 — Top 15 Hero SKUs by Revenue

In [ ]:
top = (delivered
       .merge(products[["product_id","product_name"]], on="product_id")
       .groupby(["product_id","product_name","category"])
       .agg(rev=("revenue_inr","sum"),
            units=("quantity","sum"),
            gm=("gross_margin_pct","mean"))
       .reset_index().sort_values("rev", ascending=False).head(15))
top["rev_k"] = top["rev"] / 1000

fig, ax = plt.subplots(figsize=(12, 7))
clrs = [CAT_COLORS.get(c,"#888") for c in top["category"][::-1]]
bars = ax.barh(top["product_name"][::-1].str[:35], top["rev_k"][::-1],
               color=clrs, alpha=0.88, height=0.65, edgecolor="white")
for bar, u, g in zip(bars, top["units"][::-1], top["gm"][::-1]):
    ax.text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2,
            f"₹{bar.get_width():.0f}K  |  {int(u)} units  |  {g:.0f}% GM",
            va="center", fontsize=7.5, color=DARK)
ax.set_xlabel("Revenue (₹ Thousands)")
ax.set_title("Top 15 Hero SKUs by Revenue",
             fontsize=12, fontweight="bold", color=DARK)
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="x", ls="--", alpha=0.4)
handles = [mpatches.Patch(color=CAT_COLORS[c], label=c)
           for c in top["category"].unique() if c in CAT_COLORS]
ax.legend(handles=handles, fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

print("💡 Insight: Hero SKUs should never go out of stock. Cross-check these against the reorder flag in inventory.")

## 💸 Chart 8 — Discount % vs Gross Margin (Bubble Chart)

In [ ]:
dm = (delivered.groupby(["category","channel"])
      .agg(disc=("discount_pct","mean"),
           gm=("gross_margin_pct","mean"),
           rev=("revenue_inr","sum"))
      .reset_index())
dm["size_bubble"] = dm["rev"] / dm["rev"].max() * 600

fig, ax = plt.subplots(figsize=(11, 6))
for cat, g in dm.groupby("category"):
    ax.scatter(g["disc"], g["gm"], s=g["size_bubble"],
               color=CAT_COLORS.get(cat,"#888"), alpha=0.72,
               edgecolors="white", linewidths=0.8, label=cat, zorder=3)

ax.axhline(50, ls="--", color="#aaa", lw=1.5)
ax.axvline(15, ls="--", color="#aaa", lw=1.5)
ax.text(15.3, 2,  "High Discount →", fontsize=8, color="#999", style="italic")
ax.text(0.5, 50.5, "Target Margin (50%)", fontsize=8, color="#999", style="italic")

for label, xy, bg, fc in [
    ("✅ Low Disc, High Margin",          (1, 70),  "#D5F5E3", "#27AE60"),
    ("⚠️ High Disc, High Margin",         (17, 70), "#FEF9E7", "#F39C12"),
    ("🔴 Danger: High Disc, Low Margin",  (17, 5),  "#FADBD8", RED),
]:
    ax.text(*xy, label, fontsize=8, color=fc,
            bbox=dict(boxstyle="round,pad=0.3", facecolor=bg, alpha=0.8))

ax.set_xlabel("Avg Discount %"); ax.set_ylabel("Avg Gross Margin %")
ax.set_title("Discount Rate vs Gross Margin % (Bubble = Revenue)\nQuadrant analysis by Category & Channel",
             fontsize=12, fontweight="bold", color=DARK)
ax.spines[["top","right"]].set_visible(False)
ax.legend(fontsize=7.5, ncol=2, loc="lower left")
plt.tight_layout(); plt.show()

print("💡 Insight: Bottom-right bubbles = high discount + low margin = profitability red flag. Investigate those category-channel combos.")

## 👥 Chart 9 — Customer Loyalty Segments

In [ ]:
seg_order  = ["Champion","Loyal","Potential Loyalist","Recent","At Risk","Lost"]
seg_colors = ["#1E8449","#27AE60","#3498DB","#F39C12","#E67E22","#E74C3C"]

counts = customers["loyalty_segment"].value_counts().reindex(seg_order).fillna(0)
spend  = customers.groupby("loyalty_segment")["total_spend_inr"].mean().reindex(seg_order).fillna(0)
orders_avg = customers.groupby("loyalty_segment")["total_orders"].mean().reindex(seg_order).fillna(0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Customer Loyalty Segmentation", fontsize=13, fontweight="bold", color=DARK)

for ax, data, title, fmt in [
    (axes[0], counts,      "Customer Count",          lambda x: str(int(x))),
    (axes[1], spend/1000,  "Avg Lifetime Spend (₹K)", lambda x: f"₹{x:.1f}K"),
    (axes[2], orders_avg,  "Avg Orders per Customer", lambda x: f"{x:.1f}"),
]:
    bars = ax.bar(data.index, data.values, color=seg_colors, alpha=0.88, edgecolor="white")
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+data.values.max()*0.01,
                fmt(b.get_height()), ha="center", fontsize=8.5, fontweight="bold")
    ax.set_title(title, fontweight="bold", color=DARK)
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    ax.spines[["top","right"]].set_visible(False)

plt.tight_layout(); plt.show()

champ = (customers["loyalty_segment"]=="Champion").sum() / len(customers) * 100
print(f"💡 Insight: Champions = {champ:.1f}% of base but drive the most revenue. Protect them with loyalty perks.")

## 📅 Chart 10 — Seasonality Index by Month

In [ ]:
mn_names = ["Jan","Feb","Mar","Apr","May","Jun",
            "Jul","Aug","Sep","Oct","Nov","Dec"]
mon = (delivered.groupby("month")["revenue_inr"].sum().reset_index())
mon["month_name"] = mon["month"].apply(lambda m: mn_names[m-1])
mon["index"] = (mon["revenue_inr"] / mon["revenue_inr"].mean()).round(3)
mon["color"] = mon["index"].apply(
    lambda x: RED if x>1.10 else (GOLD if x>0.88 else "#95A5A6"))

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(mon["month_name"], mon["index"], color=mon["color"],
              alpha=0.88, edgecolor="white", width=0.65)
for b, v in zip(bars, mon["index"]):
    ax.text(b.get_x()+b.get_width()/2, v+0.015, f"{v:.2f}",
            ha="center", fontsize=9, fontweight="bold", color=DARK)

ax.axhline(1.0,  ls="-",  color=DARK, lw=1.5, alpha=0.5, label="Baseline 1.0")
ax.axhline(1.10, ls="--", color=RED,  lw=1,   alpha=0.5, label="Peak (>1.10)")
ax.axhline(0.88, ls="--", color="#95A5A6", lw=1, alpha=0.5, label="Trough (<0.88)")

handles = [mpatches.Patch(color=RED,      label="Peak (>1.10)"),
           mpatches.Patch(color=GOLD,     label="Normal (0.88–1.10)"),
           mpatches.Patch(color="#95A5A6",label="Trough (<0.88)")]
ax.legend(handles=handles, fontsize=9, loc="upper left")
ax.set_title("Seasonality Index by Month\n(1.0 = annual average)",
             fontsize=12, fontweight="bold", color=DARK)
ax.set_ylabel("Seasonality Index")
ax.spines[["top","right"]].set_visible(False)
ax.set_ylim(0, mon["index"].max()*1.25)
plt.tight_layout(); plt.show()

peaks = mon[mon["index"]>1.1]["month_name"].tolist()
print(f"💡 Insight: Peak months = {peaks}. Start inventory build 10–12 weeks before these months.")

## 📣 Chart 11 — Marketing ROAS by Channel (2024)

In [ ]:
mkt24 = marketing[marketing["year"]==2024]
roas = (mkt24.groupby("marketing_channel")
        .agg(spend=("spend_inr_lakhs","sum"),
             rev=("attributed_revenue_lakhs","sum"),
             cac=("cac_inr","mean"))
        .reset_index())
roas["roas"] = (roas["rev"] / roas["spend"]).round(2)
roas = roas.sort_values("roas", ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Marketing ROI — FY2024", fontsize=13, fontweight="bold", color=DARK)

# ROAS bars
clrs_r = ["#E74C3C" if r<3 else "#F39C12" if r<4.5 else "#27AE60" for r in roas["roas"]]
bars = ax1.barh(roas["marketing_channel"], roas["roas"],
                color=clrs_r, alpha=0.88, height=0.55, edgecolor="white")
ax1.axvline(3.0, ls="--", color="#aaa", lw=1.5, label="Floor = 3×")
for b, v in zip(bars, roas["roas"]):
    ax1.text(b.get_width()+0.05, b.get_y()+b.get_height()/2,
             f"{v:.1f}×", va="center", fontsize=10, fontweight="bold", color=DARK)
ax1.set_xlabel("ROAS"); ax1.set_title("ROAS by Channel", fontweight="bold", color=DARK)
ax1.legend(fontsize=8); ax1.spines[["top","right"]].set_visible(False)

# Spend bars
rs = roas.sort_values("spend", ascending=False)
ax2.bar(rs["marketing_channel"], rs["spend"],
        color=["#E31837","#2C3E50","#FFB800","#6A0DAD","#3498DB","#27AE60","#E74C3C"][:len(rs)],
        alpha=0.88, edgecolor="white")
ax2.set_ylabel("Spend (₹ Lakhs)"); ax2.tick_params(axis="x", rotation=30, labelsize=8)
ax2.set_title("Spend Distribution", fontweight="bold", color=DARK)
ax2.spines[["top","right"]].set_visible(False)

plt.tight_layout(); plt.show()

best = roas.loc[roas["roas"].idxmax(),"marketing_channel"]
print(f"💡 Insight: {best} = highest ROAS. Shift budget from low-ROAS channels towards it.")

## 🔄 Chart 12 — Return Rate Heatmap (Category × Channel)

In [ ]:
ret = (orders.groupby(["category","channel"])
       .apply(lambda x: x["is_returned"].mean()*100)
       .reset_index(name="return_rate")
       .pivot(index="category", columns="channel", values="return_rate")
       .round(1))

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(ret, annot=True, fmt=".1f", cmap="YlOrRd",
            vmin=0, vmax=12, linewidths=0.5, linecolor="#eee",
            cbar_kws={"label":"Return Rate %","shrink":0.7}, ax=ax)
ax.set_title("Return Rate % — Category × Channel\n(Dark = High Returns = More Cost)",
             fontsize=12, fontweight="bold", color=DARK)
ax.tick_params(axis="x", rotation=30, labelsize=9)
ax.tick_params(axis="y", rotation=0, labelsize=9)
plt.tight_layout(); plt.show()

overall = orders["is_returned"].mean()*100
print(f"💡 Insight: Overall return rate = {overall:.1f}%. High-return combos = investigate sizing / product quality issues.")

## 📈 Chart 13 — YoY Category Growth (2023 vs 2024)

In [ ]:
yoy = (delivered[delivered["year"].isin([2023,2024])]
       .groupby(["year","category"])["revenue_inr"].sum()
       .reset_index()
       .pivot(index="category", columns="year", values="revenue_inr")
       .fillna(0))
yoy.columns.name = None
if 2023 in yoy.columns and 2024 in yoy.columns:
    yoy["growth"] = (yoy[2024]-yoy[2023]) / yoy[2023].replace(0,np.nan) * 100
yoy = yoy.dropna(subset=["growth"]).sort_values("growth")

fig, ax = plt.subplots(figsize=(10, 6))
clrs = [RED if v<0 else GREEN for v in yoy["growth"]]
bars = ax.barh(yoy.index, yoy["growth"], color=clrs, alpha=0.88,
               height=0.6, edgecolor="white")
ax.axvline(0, color=DARK, lw=1.5)
for bar, v in zip(bars, yoy["growth"]):
    ax.text(v+(0.5 if v>=0 else -0.5), bar.get_y()+bar.get_height()/2,
            f"{v:+.1f}%", va="center", ha="left" if v>=0 else "right",
            fontsize=9.5, fontweight="bold",
            color="#1E8449" if v>=0 else RED)
ax.set_xlabel("YoY Revenue Growth % (2023→2024)")
ax.set_title("YoY Category Revenue Growth\nGreen = Growing | Red = Declining",
             fontsize=12, fontweight="bold", color=DARK)
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="x", ls="--", alpha=0.4)
plt.tight_layout(); plt.show()

top_g = yoy["growth"].idxmax(); top_d = yoy["growth"].idxmin()
print(f"💡 Insight: Fastest growing = {top_g} | Most declining = {top_d}. Adjust buying depth accordingly.")

## 🎯 Chart 14 — Executive KPI Summary Dashboard

In [ ]:
kpis = {}
for yr in [2022, 2023, 2024]:
    d = delivered[delivered["year"]==yr]
    kpis[yr] = {
        "Revenue\n(₹L)":    round(d["revenue_inr"].sum()/100000, 1),
        "Total\nOrders":     d["order_id"].nunique(),
        "Avg Order\nValue":  round(d["selling_price"].mean(), 0),
        "Gross\nMargin %":   round(d["gross_margin_pct"].mean(), 1),
        "Avg\nDiscount %":   round(d["discount_pct"].mean(), 1),
        "Return\nRate %":    round(orders[orders["year"]==yr]["is_returned"].mean()*100, 1),
        "Unique\nBuyers":    d["customer_id"].nunique(),
    }

metrics  = list(kpis[2022].keys())
pal      = [RED, BLUE, GREEN, GOLD, "#E67E22", "#8E44AD", "#1ABC9C"]
fig      = plt.figure(figsize=(16, 7))
fig.patch.set_facecolor(DARK)

# Title strip
ax_t = fig.add_axes([0, 0.88, 1, 0.12])
ax_t.set_facecolor(RED)
ax_t.text(0.5, 0.55, "THE SOULED STORE — EXECUTIVE KPI DASHBOARD",
          ha="center", va="center", fontsize=15, fontweight="bold",
          color="white", transform=ax_t.transAxes)
ax_t.text(0.5, 0.15, "Category Analytics  |  Jayesh  |  FY2022–FY2024",
          ha="center", va="center", fontsize=9, color="#cccccc",
          transform=ax_t.transAxes, style="italic")
ax_t.axis("off")

# Mini bar per metric
n = len(metrics)
for i, (metric, col) in enumerate(zip(metrics, pal)):
    ax = fig.add_axes([0.02 + i*(0.96/n), 0.08, (0.96/n)-0.01, 0.76])
    ax.set_facecolor("#16213E")
    vals = [kpis[yr][metric] for yr in [2022,2023,2024]]
    ax.barh([2022,2023,2024], vals, color=col, alpha=0.85, height=0.5)
    for v, yr in zip(vals, [2022,2023,2024]):
        ax.text(max(vals)*0.04, yr, f"{v:,.0f}", va="center",
                fontsize=7, fontweight="bold", color="white")
    ax.set_yticks([2022,2023,2024])
    ax.set_yticklabels(["2022","2023","2024"], fontsize=7, color="white")
    ax.tick_params(axis="x", labelsize=6, colors="#aaa")
    ax.set_title(metric, fontsize=7.5, fontweight="bold", color=col, pad=5)
    for spine in ax.spines.values(): spine.set_color("#2C3E50")
    ax.grid(axis="x", ls="--", lw=0.4, color="#2C3E50")

plt.show()
print("💡 This is your one-pager — perfect as an opening slide in any interview presentation.")

---
## ✅ Analysis Complete — Key Takeaways

In [ ]:
print("=" * 60)
print("  THE SOULED STORE — KEY FINDINGS SUMMARY")
print("=" * 60)

total_rev = delivered["revenue_inr"].sum() / 100000
best_cat  = delivered.groupby("category")["revenue_inr"].sum().idxmax()
best_ch   = delivered.groupby("channel")["revenue_inr"].sum().idxmax()
best_roas = marketing.groupby("marketing_channel").apply(
    lambda x: x["attributed_revenue_lakhs"].sum() / x["spend_inr_lakhs"].sum()).idxmax()
over_pct  = (inventory["stock_health_status"]=="Overstocked").sum() / len(inventory) * 100
champ_pct = (customers["loyalty_segment"]=="Champion").sum() / len(customers) * 100

print(f"\n💰 Total Revenue (all years) : ₹{total_rev:.1f} Lakhs")
print(f"   Best Category             : {best_cat}")
print(f"   Best Channel              : {best_ch}")
print(f"   Best Festive Month Rev    : Oct / Nov")

print(f"\n📦 Inventory")
print(f"   Avg Sell-Through          : {inventory['sell_through_pct'].mean():.1f}%")
print(f"   Overstocked SKU-sizes     : {over_pct:.1f}%")
print(f"   Reorder flags raised      : {inventory['reorder_recommended'].sum()}")

print(f"\n👤 Customers")
print(f"   Total customers           : {len(customers)}")
print(f"   Champion segment          : {champ_pct:.1f}% of base")

print(f"\n📣 Marketing")
print(f"   Blended ROAS              : {marketing['attributed_revenue_lakhs'].sum()/marketing['spend_inr_lakhs'].sum():.2f}x")
print(f"   Best ROAS channel         : {best_roas}")

print(f"\n{'=' * 60}")
print("  14 charts complete — ready for interview!")
print(f"{'=' * 60}")